# Asignación de Frecuencias con Algoritmos Genéticos

Cuaderno listo para **Google Colab** o **JupyterLab**.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx


In [ ]:
n_transmisores = 8
n_frecuencias = 5

conflictos = [
    (0, 1, 1),
    (0, 2, 2),
    (1, 2, 1),
    (1, 3, 2),
    (2, 4, 1),
    (3, 4, 1),
    (3, 5, 2),
    (4, 6, 1),
    (5, 6, 1),
    (5, 7, 2),
    (6, 7, 1),
]

tam_poblacion = 100
n_generaciones = 120
prob_cruce = 0.8
prob_mutacion = 0.08
tam_torneo = 3

random.seed(42)
np.random.seed(42)


In [ ]:
G = nx.Graph()
G.add_nodes_from(range(n_transmisores))
for u, v, sep in conflictos:
    G.add_edge(u, v, sep=sep)

def crear_individuo():
    return np.random.randint(0, n_frecuencias, size=n_transmisores)

def penalizacion_total(individuo):
    penalizacion = 0
    conflictos_duros = 0
    conflictos_suaves = 0
    for u, v, sep_min in conflictos:
        diff = abs(int(individuo[u]) - int(individuo[v]))
        if diff == 0:
            penalizacion += 100
            conflictos_duros += 1
        elif diff < sep_min:
            penalizacion += 20 * (sep_min - diff)
            conflictos_suaves += 1
    return penalizacion, conflictos_duros, conflictos_suaves

def fitness(individuo):
    penalizacion, _, _ = penalizacion_total(individuo)
    return -penalizacion


In [ ]:
def seleccion_torneo(poblacion):
    candidatos = random.sample(poblacion, tam_torneo)
    return max(candidatos, key=fitness).copy()

def cruce_un_punto(padre1, padre2):
    if random.random() < prob_cruce:
        punto = random.randint(1, n_transmisores - 1)
        hijo1 = np.concatenate([padre1[:punto], padre2[punto:]])
        hijo2 = np.concatenate([padre2[:punto], padre1[punto:]])
        return hijo1, hijo2
    return padre1.copy(), padre2.copy()

def mutar(individuo):
    mutado = individuo.copy()
    for i in range(n_transmisores):
        if random.random() < prob_mutacion:
            mutado[i] = random.randint(0, n_frecuencias - 1)
    return mutado


In [ ]:
poblacion = [crear_individuo() for _ in range(tam_poblacion)]

mejores_penalizaciones = []
mejores_duros = []
mejores_suaves = []
mejor_individuo_global = None
mejor_fit_global = -float('inf')

for generacion in range(n_generaciones):
    nueva_poblacion = []
    while len(nueva_poblacion) < tam_poblacion:
        padre1 = seleccion_torneo(poblacion)
        padre2 = seleccion_torneo(poblacion)
        hijo1, hijo2 = cruce_un_punto(padre1, padre2)
        hijo1 = mutar(hijo1)
        hijo2 = mutar(hijo2)
        nueva_poblacion.append(hijo1)
        if len(nueva_poblacion) < tam_poblacion:
            nueva_poblacion.append(hijo2)
    poblacion = nueva_poblacion
    mejor_generacion = max(poblacion, key=fitness)
    mejor_fit = fitness(mejor_generacion)
    penalizacion, duros, suaves = penalizacion_total(mejor_generacion)
    mejores_penalizaciones.append(penalizacion)
    mejores_duros.append(duros)
    mejores_suaves.append(suaves)
    if mejor_fit > mejor_fit_global:
        mejor_fit_global = mejor_fit
        mejor_individuo_global = mejor_generacion.copy()

pen_final, duros_final, suaves_final = penalizacion_total(mejor_individuo_global)
print('Mejor cromosoma encontrado:', mejor_individuo_global.tolist())
print('Penalización total:', pen_final)
print('Conflictos duros:', duros_final)
print('Conflictos suaves:', suaves_final)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(mejores_penalizaciones, label='Mejor penalización')
plt.xlabel('Generación')
plt.ylabel('Penalización')
plt.title('Evolución del algoritmo genético')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(mejores_duros, label='Conflictos duros')
plt.plot(mejores_suaves, label='Conflictos suaves')
plt.xlabel('Generación')
plt.ylabel('Cantidad')
plt.title('Evolución de conflictos')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)
palette = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'yellow']
node_colors = [palette[int(f) % len(palette)] for f in mejor_individuo_global]
nx.draw(G, pos, with_labels=True, node_color=node_colors, node_size=900, font_size=10, edgecolors='black')
edge_labels = {(u, v): f'sep>={sep}' for u, v, sep in conflictos}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)
plt.title('Asignación final de frecuencias')
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.bar([f'T{i}' for i in range(n_transmisores)], mejor_individuo_global)
plt.xlabel('Transmisor')
plt.ylabel('Frecuencia asignada')
plt.title('Frecuencia por transmisor')
plt.grid(axis='y')
plt.show()
